# Q3 — Naive Bayes ("Spam vs Ham" style)

**Adaptation note:** the original lab prompt asks for spam/ham email classification,
but per lab instructions every question must use the BSDS500-derived dataset only.
We treat this as the same kind of problem Naive Bayes is classically used for: binary
classification from a feature vector under a conditional independence assumption —
here **"edge"** plays the role of *spam* and **"non-edge"** the role of *ham*, with
pixel colour / gradient / texture features in place of word/token features.

## Setup

This notebook expects to be run from the project root, alongside a `common.py`
(or with the helper cell below), and with the following layout already in place:

```
project_root/
├── archive/                # BSDS500 images + ground_truth
├── data/bsds_features.csv  # produced by the feature-extraction notebook
├── results/figures/
└── results/metrics/
```

If you don't have a `common.py` file in this directory, run the cell below first —
it defines the same `load_split` / `save_metrics` helpers used across all seven
questions so this notebook is self-contained.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve()
DATA_CSV = ROOT / "data" / "bsds_features.csv"
FIG_DIR = ROOT / "results" / "figures"
METRIC_DIR = ROOT / "results" / "metrics"
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = ["R", "G", "B", "gray", "grad_mag", "grad_dir",
                "laplacian", "local_std", "x_norm", "y_norm"]
LABEL_COL = "is_edge"
RANDOM_STATE = 42


def load_split(test_size=0.2, scale=True):
    df = pd.read_csv(DATA_CSV)
    X = df[FEATURE_COLS].values.astype(np.float64)
    y = df[LABEL_COL].values.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y
    )

    if scale:
        mu, sigma = X_train.mean(axis=0), X_train.std(axis=0)
        sigma[sigma == 0] = 1.0
        X_train = (X_train - mu) / sigma
        X_test = (X_test - mu) / sigma

    return X_train, X_test, y_train, y_test


def save_metrics(name, d):
    path = METRIC_DIR / f"{name}.json"
    with open(path, "w") as f:
        json.dump(d, f, indent=2, default=float)
    print(f"saved metrics -> {path}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report, RocCurveDisplay)

## Load data
GaussianNB is not scale-sensitive, but scaling keeps things consistent with the other questions and doesn't change its predictions.

In [ ]:
X_train, X_test, y_train, y_test = load_split(scale=False)
print(f"train={X_train.shape}, test={X_test.shape}")

## Fit Gaussian Naive Bayes

In [ ]:
clf = GaussianNB()
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
probs = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, preds)
cm = confusion_matrix(y_test, preds)
report = classification_report(y_test, preds, target_names=["ham (non-edge)", "spam (edge)"])
print(f"Test accuracy: {acc:.4f}")
print(report)

## Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Greens")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1]); ax.set_xticklabels(["ham", "spam"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["ham", "spam"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Naive Bayes Confusion Matrix")
plt.tight_layout()
plt.savefig(FIG_DIR / "q3_nb_confusion_matrix.png", bbox_inches="tight")
plt.show()

## ROC curve

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
RocCurveDisplay.from_predictions(y_test, probs, ax=ax, name="GaussianNB")
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_title("Naive Bayes ROC Curve")
plt.tight_layout()
plt.savefig(FIG_DIR / "q3_nb_roc.png", bbox_inches="tight")
plt.show()

## Parameter modification: sensitivity to `var_smoothing`

In [ ]:
smoothing_values = [1e-11, 1e-9, 1e-7, 1e-5, 1e-3, 1e-1, 1.0]
smoothing_accs = []
for vs in smoothing_values:
    c = GaussianNB(var_smoothing=vs)
    c.fit(X_train, y_train)
    a = accuracy_score(y_test, c.predict(X_test))
    smoothing_accs.append(a)
    print(f"var_smoothing={vs:<9g} accuracy={a:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([str(v) for v in smoothing_values], smoothing_accs, marker="o")
ax.set_xlabel("var_smoothing")
ax.set_ylabel("test accuracy")
ax.set_title("Naive Bayes: accuracy vs var_smoothing")
plt.tight_layout()
plt.savefig(FIG_DIR / "q3_smoothing_sensitivity.png", bbox_inches="tight")
plt.show()

## Save metrics

In [ ]:
save_metrics("q3_naive_bayes", {
    "test_accuracy": acc,
    "confusion_matrix": cm.tolist(),
    "classification_report": report,
    "class_priors": clf.class_prior_.tolist(),
    "smoothing_sweep": {"var_smoothing": smoothing_values, "accuracies": smoothing_accs},
})